# Table Keypoint Detection — YOLOv11-nano-pose

Train a YOLO pose model to detect 6 table keypoints (4 corners + 2 net points).

**Input data:** `table_keypoints.zip` (640×640 images + YOLO keypoint labels)
**Output:** YOLO pose model + ONNX export

## How to use:
1. Upload `table_keypoints.zip` to Google Drive root
2. Run each cell in order
3. Download the exported model at the end

## Cell 1: Mount Drive & unzip data

In [ ]:
!pip install -q ultralytics

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/table_keypoints.zip /content/
!unzip -qo /content/table_keypoints.zip -d /content/

print('\nDataset contents:')
!ls /content/table_keypoints/
!echo 'Train:' && ls /content/table_keypoints/train/images/ | wc -l
!echo 'Val:' && ls /content/table_keypoints/val/images/ | wc -l
!cat /content/table_keypoints/dataset.yaml

## Cell 2: Fix dataset.yaml path for Colab

In [ ]:
# Update path in dataset.yaml to point to Colab location
import yaml
from pathlib import Path

yaml_path = Path('/content/table_keypoints/dataset.yaml')
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = '/content/table_keypoints'

with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Updated dataset.yaml:')
!cat /content/table_keypoints/dataset.yaml

## Cell 3: Verify — visualize samples with keypoints

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random
import numpy as np

IMG_SIZE = 640
NUM_KP = 6
COLORS = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6', '#e91e63']

train_imgs = sorted(Path('/content/table_keypoints/train/images').glob('*.jpg'))
sample = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)

    txt_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    if txt_path.exists():
        parts = txt_path.read_text().strip().split()
        # Skip class + bbox (5 values), then keypoints (x, y, v) * 6
        for i in range(NUM_KP):
            idx = 5 + i * 3
            x, y, v = float(parts[idx]), float(parts[idx+1]), float(parts[idx+2])
            if v > 0:
                ax.plot(x * IMG_SIZE, y * IMG_SIZE, 'o', color=COLORS[i], markersize=8, markeredgecolor='white', markeredgewidth=1.5)
                ax.annotate(str(i+1), (x*IMG_SIZE+5, y*IMG_SIZE-5), color=COLORS[i], fontsize=9, fontweight='bold')

    ax.set_title(img_path.stem, fontsize=8)
    ax.axis('off')

plt.suptitle('Table keypoints: 1-4=corners (clockwise from far-left), 5-6=net', fontsize=13)
plt.tight_layout()
plt.show()

## Cell 4: Train YOLOv11-nano-pose

- `model=yolo11n-pose.pt` — nano pose model pretrained on COCO keypoints
- `imgsz=640` — matches training images
- `epochs=200` — small dataset needs more epochs
- `patience=30` — early stopping

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')

results = model.train(
    data='/content/table_keypoints/dataset.yaml',
    epochs=200,
    imgsz=640,
    batch=16,
    patience=30,
    device=0,
    project='/content/yolo_runs',
    name='table_kp_v1',
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.2,
    hsv_v=0.2,
    flipud=0.0,
    fliplr=0.0,       # no flip — left/right keypoints would swap
    mosaic=0.3,       # light mosaic
    mixup=0.0,
    scale=0.3,
    translate=0.1,
    perspective=0.0005,  # slight perspective warp
)

## Cell 5: Training curves & metrics

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path('/content/yolo_runs/table_kp_v1')

for img_name in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    img_path = run_dir / img_name
    if img_path.exists():
        print(f'\n{img_name}:')
        display(Image(filename=str(img_path), width=700))

## Cell 6: Validate & visualize predictions

In [ ]:
import math

best_model = YOLO(str(run_dir / 'weights' / 'best.pt'))
val_results = best_model.val(data='/content/table_keypoints/dataset.yaml', imgsz=640)

print(f'\n=== Validation Results ===')
print(f'mAP50:     {val_results.box.map50:.4f}')
print(f'mAP50-95:  {val_results.box.map:.4f}')
print(f'Precision: {val_results.box.mp:.4f}')
print(f'Recall:    {val_results.box.mr:.4f}')

# Visualize predictions
val_imgs = sorted(Path('/content/table_keypoints/val/images').glob('*.jpg'))
sample_imgs = random.sample(val_imgs, min(12, len(val_imgs)))

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
for ax, img_path in zip(axes.flatten(), sample_imgs):
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)

    # Ground truth
    txt_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    if txt_path.exists():
        parts = txt_path.read_text().strip().split()
        for i in range(NUM_KP):
            idx = 5 + i * 3
            x, y, v = float(parts[idx]), float(parts[idx+1]), float(parts[idx+2])
            if v > 0:
                ax.plot(x*IMG_SIZE, y*IMG_SIZE, 'o', color='lime', markersize=6, markeredgecolor='white', markeredgewidth=1)

    # Predictions
    preds = best_model(img_bgr, imgsz=640, conf=0.25, verbose=False)[0]
    if preds.keypoints is not None and len(preds.keypoints) > 0:
        kps = preds.keypoints[0].data[0].cpu().numpy()  # (6, 3)
        for i, (x, y, conf) in enumerate(kps):
            if conf > 0.3:
                ax.plot(x, y, 'x', color='red', markersize=8, markeredgewidth=2)
                ax.annotate(str(i+1), (x+3, y-3), color='red', fontsize=8)

    ax.set_title(img_path.stem, fontsize=8)
    ax.axis('off')

fig.suptitle('Green=GT, Red=Predicted', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Export to ONNX

In [ ]:
import os

best_model.export(format='onnx', imgsz=640, simplify=True)

print('\nExported models:')
for f in (run_dir / 'weights').glob('*'):
    sz = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {sz:.1f} MB')

## Cell 8: Save to Google Drive

In [ ]:
import shutil

drive_dir = Path('/content/drive/MyDrive/yolo_table_keypoints')
drive_dir.mkdir(exist_ok=True)

# Copy weights
for f in (run_dir / 'weights').glob('*'):
    shutil.copy(f, drive_dir / f.name)

# Copy results
if (run_dir / 'results.png').exists():
    shutil.copy(run_dir / 'results.png', drive_dir / 'results.png')

print(f'Saved to: {drive_dir}')
!ls -lh {drive_dir}